# easy_ViTPose webapp — full Colab runner

Runs the **whole webapp** here (pose, shuttlecock, court tracking, TrackNetV3) on Colab's free GPU — same code as local, just faster (especially TrackNetV3, which is PyTorch/CUDA-based and was slow on CPU).

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.

Run every cell top to bottom, in order. Cell 2 will ask you to upload a zip file (`easy_vitpose_colab_code.zip`) — this is the project's source code (no model weights, no secrets), generated alongside this notebook.

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('\nWARNING: no GPU detected. Go to Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart session.')


## 2. Upload the project code

Upload `easy_vitpose_colab_code.zip` (source code only — `easy_ViTPose/`, `webapp/`, `tracknet_v3/`, plus the root `requirements*.txt` — no model weights, no `.env`/API keys).

In [ ]:
import os
from google.colab import files

print('Select easy_vitpose_colab_code.zip...')
uploaded = files.upload()
zip_name = next(iter(uploaded.keys()))

PROJECT_DIR = '/content/easy_ViTPose-main'
os.makedirs(PROJECT_DIR, exist_ok=True)
!unzip -q -o "{zip_name}" -d "{PROJECT_DIR}"
%cd {PROJECT_DIR}
!ls


## 3. Install dependencies

In [ ]:
# requirements.txt pins onnxruntime==1.16.0, which isn't published for Colab's (newer) Python --
# that single unresolvable pin fails the WHOLE `pip install -r` command (nothing after it in the
# file gets installed either, e.g. ultralytics). Strip that one line out; a current onnxruntime
# build is installed separately right below instead (any recent version works fine for inference).
!grep -v '^onnxruntime==' requirements.txt > /tmp/requirements_colab.txt
!pip install -q -r /tmp/requirements_colab.txt
!pip install -q flask parse gdown huggingface_hub

# GPU build of onnxruntime for faster ViTPose inference (falls back to the CPU build if this errors).
!pip install -q onnxruntime-gpu || pip install -q onnxruntime

!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
print('Dependencies installed.')


## 4. Download AI models

ViTPose + YOLOv8 (pose) from the original `easy_ViTPose` / `Ultralytics` Hugging Face repos, and the pretrained TrackNetV3 checkpoints (shuttlecock trajectory tracking) from the paper authors' Google Drive release.

In [ ]:
import os

MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

# ViTPose-L (COCO = human, AP10K = animal), ONNX -- filenames must match webapp/app.py's MODEL_CONFIG
!wget -q -O "{MODELS_DIR}/vitpose-l-coco.onnx" https://huggingface.co/JunkyByte/easy_ViTPose/resolve/main/onnx/coco/vitpose-l-coco.onnx
!wget -q -O "{MODELS_DIR}/vitpose-l-ap10k.onnx" https://huggingface.co/JunkyByte/easy_ViTPose/resolve/main/onnx/ap10k/vitpose-l-ap10k.onnx
# YOLOv8-L (person/animal detector that runs ahead of ViTPose)
!wget -q -O "{MODELS_DIR}/yolov8l.pt" https://huggingface.co/Ultralytics/YOLOv8/resolve/main/yolov8l.pt

print('ViTPose + YOLOv8 done:')
!ls -lh "{MODELS_DIR}"


In [ ]:
import os

TRACKNET_DIR = os.path.join(MODELS_DIR, 'tracknet')
os.makedirs(TRACKNET_DIR, exist_ok=True)

# Pretrained TrackNetV3 checkpoints (TrackNet_best.pt + InpaintNet_best.pt), released by the
# paper authors (qaz812345/TrackNetV3) at this Google Drive file id.
!gdown 1CfzE87a0f6LhBp0kniSl1-89zaLCZ8cA -O /content/TrackNetV3_ckpts.zip
!unzip -q -o /content/TrackNetV3_ckpts.zip -d /content/tracknet_ckpts_tmp
# Zip's internal layout is ckpts/TrackNet_best.pt + ckpts/InpaintNet_best.pt (confirmed from the release archive)
!mv /content/tracknet_ckpts_tmp/ckpts/*.pt "{TRACKNET_DIR}/"
!rm -rf /content/tracknet_ckpts_tmp /content/TrackNetV3_ckpts.zip

print('TrackNetV3 checkpoints:')
!ls -lh "{TRACKNET_DIR}"


## 5. Roboflow API key

Needed for the **Shuttlecock**, **Gabungan**, and **Posisi Lapangan** modes (they call Roboflow-hosted models). Not needed for plain pose modes or TrackNetV3 (fully local).

Get a *Private* API key at [app.roboflow.com/settings/api](https://app.roboflow.com/settings/api). This is typed into a masked prompt, not saved in this notebook file.

In [ ]:
import getpass, os

api_key = getpass.getpass('Roboflow Private API key (leave blank to skip): ').strip()
env_path = os.path.join(PROJECT_DIR, 'webapp', '.env')
if api_key:
    with open(env_path, 'w') as f:
        f.write(f'ROBOFLOW_API_KEY={api_key}\n')
    print('Saved to webapp/.env')
else:
    print('Skipped -- Shuttlecock/Gabungan/Posisi Lapangan modes will show a "belum di-set" error until you re-run this cell with a key.')


## 6. Start the webapp + public URL

Runs `webapp/app.py` in a background thread (so this cell returns immediately) and opens a public tunnel to it via `localtunnel` (no account/signup needed, unlike ngrok). The very first time you open the printed URL, `localtunnel` shows a one-time interstitial page — click "Click to Continue".

In [ ]:
import subprocess, threading, time, os, urllib.request

os.chdir(PROJECT_DIR)
flask_log_path = '/content/flask.log'
flask_log = open(flask_log_path, 'w')

def _run_flask():
    subprocess.run(['python', 'webapp/app.py'], stdout=flask_log, stderr=subprocess.STDOUT)

flask_thread = threading.Thread(target=_run_flask, daemon=True)
flask_thread.start()

# Actually verify the port comes up, instead of assuming it will after a fixed sleep --
# a crash on import (e.g. a missing dependency) shows up here immediately.
started = False
for _ in range(20):
    time.sleep(1)
    try:
        code_ = urllib.request.urlopen('http://127.0.0.1:5050/', timeout=2).getcode()
        started = True
        break
    except Exception:
        continue

if started:
    print('Flask is up on port 5050.')
else:
    print('Flask did NOT come up after 20s. Log so far:\n')
    with open(flask_log_path) as f:
        print(f.read())


In [ ]:
# One-time: install localtunnel (needs Node/npm, both preinstalled on Colab)
!npm install -g localtunnel --silent


In [ ]:
import subprocess, time, re, urllib.request

# Public IP -- localtunnel's interstitial page asks for this as the tunnel "password"
try:
    public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com', timeout=10).read().decode().strip()
    print('Tunnel password (if asked):', public_ip)
except Exception as e:
    print('Could not fetch public IP:', e)

lt_log = open('/content/lt.log', 'w')
lt_proc = subprocess.Popen(['lt', '--port', '5050'], stdout=lt_log, stderr=subprocess.STDOUT)
time.sleep(5)

with open('/content/lt.log') as f:
    log_content = f.read()
print(log_content)

match = re.search(r'(https://\S+\.loca\.lt)', log_content)
if match:
    print('\n=========================================')
    print('Webapp URL:', match.group(1))
    print('=========================================')
else:
    print('\nURL not found yet -- re-run this cell, or check /content/lt.log')


## Notes

- **TrackNetV3** now auto-detects the GPU (`tracknet_v3/predict.py` was patched for this) -- on Colab's T4 it should run far faster than the CPU run from earlier in this project.
- Re-running cell 6's Flask-start cell after it's already running will fail to bind port 5050 (address in use) -- only run it once per session; if you need to restart, `Runtime -> Restart session` first.
- Files uploaded/generated during processing live under `webapp/static/uploads/` and `webapp/static/results/` inside this Colab VM -- they disappear when the session ends. Download anything you want to keep.
- Colab free-tier sessions disconnect after a period of inactivity / a fixed max duration -- this is a Colab limit, not something in this notebook's control.